# OpenPlaque Notebook 08: Volume-Based Boundary Tuning

This notebook runs from a clean Colab environment.

It tunes boundary refinement using **minimum component volume in mm³**, not fixed voxel count.

Research use only. Not for clinical decision-making.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque || true
!git -C /content/OpenPlaque pull


In [ ]:
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt
!pip -q install gdown


In [ ]:
import os, sys
from pathlib import Path

repo = Path("/content/OpenPlaque")
src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

for d in [os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]]:
    os.makedirs(d, exist_ok=True)

print("OpenPlaque path:", src)


In [ ]:
!nvidia-smi


In [ ]:
import zipfile
from pathlib import Path

model_zip = Path("/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip")
model_target = Path("/content/nnUNet_results/Dataset001_CCTA_DHM")

if model_target.exists():
    print("Model already extracted:", model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f"Missing model ZIP: {model_zip}")
    with zipfile.ZipFile(model_zip) as z:
        z.extractall("/content/nnUNet_results")
    print("Extracted model")


In [ ]:
from pathlib import Path

sample_root = Path("/content/sample_dataset/Sample_Dataset")
if sample_root.exists():
    print("Sample dataset already present:", sample_root)
else:
    !mkdir -p /content/sample_dataset
    !gdown --folder "https://drive.google.com/drive/folders/1i4XD-GfP-wS50m9smGR1LzBRJokKro9_?usp=sharing" -O /content/sample_dataset --remaining-ok

!find /content/sample_dataset -maxdepth 3 | head -60


In [ ]:
from openplaque.boundary_volume import list_sample_cases

cases = list_sample_cases("/content/sample_dataset/Sample_Dataset")
print("Cases:", len(cases))
print(cases[:3])


## Quick focused test

In [ ]:
from openplaque.boundary_volume import (
    focused_parameter_grid,
    tune_volume_based_refinement,
    summarize_volume_tuning,
)

quick_grid = focused_parameter_grid(
    min_component_volume_mm3=(0, 1, 2, 3, 5),
    lumen_distance_voxels=(0,),
    high_hu_threshold=(None,),
    low_hu_threshold=(None,),
)

df_test = tune_volume_based_refinement(
    sample_root="/content/sample_dataset/Sample_Dataset",
    pred_dir="/content/openplaque_predictions",
    max_cases=3,
    grid=quick_grid,
    overwrite_predictions=False,
)

summary_test = summarize_volume_tuning(df_test)
summary_test.head(10)


## Full focused tuning

This intentionally avoids erosion and uses a narrower, interpretable parameter space.


In [ ]:
full_grid = focused_parameter_grid(
    min_component_volume_mm3=(0, 0.5, 1, 2, 3, 5, 8, 10, 15, 20),
    lumen_distance_voxels=(0, 1),
    high_hu_threshold=(None,),
    low_hu_threshold=(None,),
)

df = tune_volume_based_refinement(
    sample_root="/content/sample_dataset/Sample_Dataset",
    pred_dir="/content/openplaque_predictions",
    max_cases=None,
    grid=full_grid,
    overwrite_predictions=False,
)

summary = summarize_volume_tuning(df)
summary.head(20)


In [ ]:
import os

out_dir = "/content/drive/MyDrive/OpenPlaque/Tuning"
os.makedirs(out_dir, exist_ok=True)

df.to_csv(f"{out_dir}/volume_based_boundary_case_results.csv", index=False)
summary.to_csv(f"{out_dir}/volume_based_boundary_parameter_summary.csv", index=False)

with open(f"{out_dir}/best_volume_based_boundary_refinement.txt", "w") as f:
    f.write("Best volume-based boundary refinement parameters\n")
    f.write(str(summary.iloc[0]))
    f.write("\n")

print("Saved tuning outputs to:", out_dir)
print("\nBest parameters:")
print(summary.iloc[0])


## Raw baseline vs best volume-based refinement

In [ ]:
raw = summary[
    (summary["min_component_volume_mm3"] == 0)
    & (summary["lumen_distance_voxels"] == 0)
].head(1)

best = summary.head(1)

print("RAW BASELINE")
print(raw.T)

print("\nBEST VOLUME-BASED REFINEMENT")
print(best.T)


## Candidate conservative defaults

A good default should improve volume error without zeroing too many cases.


In [ ]:
display_cols = [
    "min_component_volume_mm3",
    "lumen_distance_voxels",
    "mean_dice",
    "mean_precision",
    "mean_recall",
    "mean_abs_volume_error_mm3",
    "median_abs_volume_error_mm3",
    "mean_removed_fraction",
    "zeroed_cases",
    "mean_score",
]

summary[display_cols].head(30)
